In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
import seaborn as sns
sns.set(style='whitegrid')

In [ ]:
dataset='finewebtext'
scion_method='constrained_scion' 
root='out'
parameterization_folder=f"{scion_method}_out"
alpha=0.5
epochs=1000

df_dict = {} # {parameterization: {job_name: df}}
parameterization = parameterization_folder
df_dict[parameterization] = {}

for job_name in os.listdir(os.path.join(root, parameterization)):
        # print(job_name)
    if f'alpha_skipconnection_{alpha}' in job_name:
        log_file = os.path.join(root, parameterization, job_name, 'log.csv')

        if os.path.exists(log_file):
            df_dict[parameterization][job_name] = pd.read_csv(log_file, header=1)
        else:
            print(f'{log_file} does not exist')


## One plot per lr, with multiple subplots per layertype.

In [ ]:

class MplColorHelper:

    def __init__(self, cmap_name, start_val, stop_val):
        self.cmap_name = cmap_name
        self.cmap = plt.get_cmap(cmap_name)
        self.norm = mpl.colors.Normalize(vmin=start_val, vmax=stop_val)
        self.scalarMap = cm.ScalarMappable(norm=self.norm, cmap=self.cmap)

    def get_rgb(self, val):
        return self.scalarMap.to_rgba(val)

layer_types = [
    ('token_embedding_act_abs_mean', 'Word Embedding', (1.5e-2, 3.5e-2)),
    ('attn_act_abs_mean', 'Attention Output', (1e-2, 1e4)),
    ('mlp_act_abs_mean', 'FFN Output', (1e-2, 1e4)),
    ('last_layer_act_abs_mean', 'Final Residual Stream', (1e-2, 1e2)),
    ('lm_head_act_abs_mean', 'Output Logits', (5e-2, 1e2)),
]

if scion_method=='unconstrained_scion':
    parameterizations = [
        (f'{scion_method}', r'Unconstrained Scion', 'Spectral'),
    ]
elif scion_method=='projected_scion':
    parameterizations = [
        (f'{scion_method}', r'Projected Scion', 'Spectral'),
        ]
    
elif scion_method=='constrained_scion':
    parameterizations = [
        (f'{scion_method}', r'Scion', 'Spectral'),
        ]
    

seeds = [1,2,3]
depths = [2, 4, 8, 16, 32, 64]
lr_list = [0.001953125, 0.0009765625, 0.00048828125, 0.000244140625, 0.0001220703125, 6.103515625e-05]

t_max = 9

color_helper = MplColorHelper('coolwarm', 0, t_max)
n_cols = len(parameterizations)
n_rows = len(layer_types)


for lr in lr_list:
    for parameterization_idx, (parameterization, parameterization_str, extra) in enumerate(parameterizations):
        results_matrix = np.zeros((len(layer_types), t_max, len(depths), len(seeds), len(parameterizations))) # (layer_type, t, width, seed)
        for depth_idx, depth in enumerate(depths):
            for seed_idx, seed in enumerate(seeds):
                depth_str = float(depth) if depth != 0 else int(depth)
                job_name = f'hptransferdepth_alpha_skipconnection_{alpha}_spectral_sign__depth{depth}_seed{seed}_lr{lr}'
                try:
                    ckpt_df = df_dict[f'{parameterization}_out'][job_name]
                except:
                    print(parameterization, job_name)

                for layer_type_idx, (layer_type, layer_type_str, ylims) in enumerate(layer_types):
                    results_matrix[layer_type_idx, :, depth_idx, seed_idx, parameterization_idx] = ckpt_df[layer_type].dropna().values[:t_max].flatten()

        for layer_type_idx, (layer_type, layer_type_str, ylims) in enumerate(layer_types):
            fig, axes = plt.subplots(1, 1, figsize=(10, 6))
            axes = np.array(axes).reshape((1, 1))
            ax = axes[0, 0]
            for t in range(0,t_max):
                means = []
                stderrs = []
                for depth_idx, depth in enumerate(depths):
                    nnz_results = results_matrix[layer_type_idx, t, depth_idx][results_matrix[layer_type_idx, t, depth_idx] != 0]
                    means.append(nnz_results.mean())
                    stderrs.append(np.std(nnz_results, ddof=1) / np.sqrt(len(nnz_results)))
                means = np.array(means)
                stderrs = np.array(stderrs)
                ax.plot(depths, means, label=f'{t+1}', color=color_helper.get_rgb(t), marker='.')
                ax.fill_between(depths, means-stderrs, means+stderrs, color=color_helper.get_rgb(t), alpha=0.5)

            ax.set_title(parameterization_str + f" with alpha={alpha}\n")
            ax.set_xlabel('Depth (L)')
            ax.set_xscale('log', base=2)
            ax.set_ylabel(layer_type_str)
            ax.set_yscale('log')
            if parameterization_idx > 0:
                ax.yaxis.set_ticklabels([])

            axes[0, 0].legend(loc='upper left', fontsize=8, title='Step')

            plt.tight_layout()
            # plt.savefig(os.path.join("coord_checks_figs", f"alpha_{alpha}_{scion_method}",f'coord_check_fig_{scion_method}_{dataset}_alpha_{alpha}_lr_{lr}_{layer_type_str}.pdf'))
            plt.show()
            plt.close()